<a href="https://colab.research.google.com/github/rybak97/Gemini_models_and_agents/blob/main/2_Tokens%2C_thinking%2C_multimodal_prompts%2C_media_resolution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import userdata
from google import genai
from google.genai import types

from IPython.display import Markdown

In [4]:
%pip install -U -q "google-genai>=2.9.0" # 2.9 for the latest interactions API additions

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 862.0 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.4/262.4 kB 23.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.58.0 which is incompatible.


In [2]:
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')

In [3]:
client = genai.Client(api_key=GEMINI_API_KEY)

In [5]:
MODEL_ID = "gemini-3-flash-preview"

# Thinking process

All Gemini models are trained to do a thinking process (or reasoning) before getting to a final answer. As a result, those models usually get better results on harder tasks that require multiple processing steps: complex math, coding, reasoning over multi-step instructions and multimodal understanding.

While thinking is always on, you can configure the amount of thinking the model does by using thinking levels (minimal, low, medium, high). This lets you balance between response speed/cost and reasoning depth depending on your use case.

# Understanding thinking models

Thinking models are optimized for complex tasks that need multiple rounds of strategizing and iteratively solving.

You can control the thinking effort using **thinking levels**:

| Thinking Level | Use Case |
|---|---|
| **High** (default) | Complex reasoning, math, coding, multi-step problems |
| **Medium** | Good balance between speed and reasoning |
| **Low** | Faster responses with some reasoning |
| **Minimal** | Fastest responses, minimal reasoning (roughly equivalent to "off") |


# Using thinking levels

In [6]:
prompt = """
    You are playing the 20 question game. You know that what you are looking for
    is a aquatic mammal that doesn't live in the sea, is venomous and that's
    smaller than a cat. What could that be and how could you make sure?
"""

interaction = client.interactions.create(
    model=MODEL_ID,
    input=prompt,
)

display(Markdown(interaction.output_text))

Based on your description, the animal you are looking for is the **Platypus** (*Ornithorhynchus anatinus*).

### Why it fits your criteria:
1.  **Aquatic (Non-Sea):** The platypus is semi-aquatic but lives exclusively in freshwater systems, such as rivers, lakes, and streams in eastern Australia and Tasmania.
2.  **Venomous:** It is one of the very few venomous mammals. Male platypuses have a calcified spur on each hind ankle connected to a venom gland. While not lethal to humans, the venom causes excruciating pain.
3.  **Smaller than a cat:** An adult platypus typically weighs between 1.5 and 5 pounds and measures about 15 to 20 inches in length. For comparison, the average domestic cat weighs 8 to 10 pounds and is roughly the same length or longer.

***

### How could you make sure?
If you were looking at the animal in the wild (or a specimen), you could verify its identity through these unique physiological markers:

*   **The "Frankenstein" Anatomy:** Look for a "duck-bill" (which is actually soft and leathery), a beaver-like tail, and otter-like fur. No other mammal has this specific combination.
*   **The Hind Legs:** To confirm it is the venomous version, you would look for the **tarsal spurs** located on the inside of the hind ankles. (Note: Only males have functional venomous spurs; females lose their rudimentary spurs within their first year).
*   **Check for Nipples:** Unlike almost all other mammals, the platypus has no teats. It secretes milk through pores in its skin, which the young lap up from its belly.
*   **Egg-laying:** As a monotreme, it lays leathery eggs similar to a reptile. If the mammal is guarding a nest of eggs in a riverbank burrow, it is definitely a platypus (or its cousin, the echidna, but echidnas aren't aquatic).

***

### An Alternative Candidate: The Water Shrew
If the platypus feels "too big," the only other animal that fits is the **European Water Shrew** (*Neomys fodiens*).
*   **Why it fits:** It is much smaller than a cat (mouse-sized), lives in freshwater, and has **venomous saliva** that it uses to paralyze prey like frogs and fish. 
*   **How to make sure:** You would look for red-tipped teeth (a characteristic of some shrews) and check for a fringe of stiff hairs on the feet and tail that act like "flippers" for swimming.

In [7]:
print("Prompt tokens:", interaction.usage.total_input_tokens)
print("Thoughts tokens:", interaction.usage.total_thought_tokens)
print("Output tokens:", interaction.usage.total_output_tokens)
print("Total tokens:", interaction.usage.total_tokens)

Prompt tokens: 62
Thoughts tokens: 941
Output tokens: 585
Total tokens: 1588


In [8]:
interaction = client.interactions.create(
    model=MODEL_ID,
    input=prompt,
    generation_config={
        "thinking_level": "low",
    },
)

display(Markdown(interaction.output_text))

Based on the clues provided, the animal you are looking for is the **Platypus** (*Ornithorhynchus anatinus*).

Here is why it fits all your criteria:
1.  **Aquatic mammal that doesn't live in the sea:** The platypus is a semi-aquatic mammal endemic to eastern Australia, including Tasmania. It lives in freshwater rivers, lakes, and streams.
2.  **Venomous:** It is one of the few venomous mammals. While both sexes are born with ankle spurs, only the **male** has spurs that produce a potent venom capable of causing severe pain to humans and killing smaller animals.
3.  **Smaller than a cat:** While people often imagine them to be large, an average platypus is only about 15 to 20 inches (38 to 50 cm) long and weighs between 1.5 to 5 lbs (0.7 to 2.4 kg). A standard domestic cat usually weighs between 8 and 10 lbs.

***

### How could you make sure?
If you were playing 20 Questions and wanted to confirm this identity with 100% certainty, you should ask these three targeted questions:

1.  **"Does it lay eggs?"**
    The platypus is a **monotreme** (an egg-laying mammal). This immediately narrows the field down to just the platypus and the echidna. Since echidnas are terrestrial and not venomous to humans, the answer would have to be the platypus.
2.  **"Does it have a bill like a duck and a tail like a beaver?"**
    This is the most iconic physical description of the platypus. When the first specimen was sent to Europe, scientists actually thought it was a hoax made of different animals sewn together.
3.  **"Does it use electrolocation to find its prey?"**
    The platypus is the only mammal known to have a sense of electroreception. It closes its eyes, ears, and nose underwater and detects the tiny electrical signals produced by the muscular contractions of its prey (like shrimp or larvae).

**Note on a "Scientific" Alternative:**
Technically, the **Eurasian Water Shrew** (*Neomys fodiens*) also fits your description. It is a semi-aquatic freshwater mammal, it is much smaller than a cat, and it has venomous saliva used to paralyze prey. However, because the platypus's venom is more "famous" and it is more strictly aquatic, it is almost always the intended answer for this riddle!

In [9]:
print("Prompt tokens:", interaction.usage.total_input_tokens)
print("Thoughts tokens:", interaction.usage.total_thought_tokens)
print("Output tokens:", interaction.usage.total_output_tokens)
print("Total tokens:", interaction.usage.total_tokens)

Prompt tokens: 62
Thoughts tokens: 660
Output tokens: 563
Total tokens: 1285


# Multimodal prompts

Gemini models have strong multimodal understanding capabilities. You can include text, PDF, audio and videos in your prompt requests and get text or code responses.

In [10]:
interaction = client.interactions.create(
    model=MODEL_ID,
    input=[
        {
            "type": "video",
            "uri": "https://youtu.be/DSxM6-u2LSs?si=u8fk_rq6xj2tQDpt",
        },
        {"type": "text", "text": "What do you see in this video? Describe it briefly."},
    ],
)

(Markdown(interaction.output_text))

The video features a woman with long black hair, a white hood, a red soccer jersey, and black pants dancing and singing in a cemetery. The video is stylized with rapid cuts and features scenes of the woman in front of a mausoleum and on a path through the cemetery. There are also several shots of angel statues and tombstones.

# Media resolution

You can specify a media resolution, which controls how images are tokenized and how many tokens are used. This can be controlled **per file**.

| Resolution | Images | PDFs | Video |
|---|---|---|---|
| `MEDIA_RESOLUTION_HIGH` | 1120 tokens | 1120 tokens | 280 tokens/frame |
| `MEDIA_RESOLUTION_MEDIUM` | 560 tokens | 560 tokens (default for PDFs) | 70 tokens/frame |
| `MEDIA_RESOLUTION_LOW` | 280 tokens | 280 tokens | 70 tokens/frame |
| `MEDIA_RESOLUTION_UNSPECIFIED` (default) | Same as HIGH for images | Same as MEDIUM for PDFs | Same as MEDIUM for video |

Note that these are maximums, and the actual token usage will usually be slightly lower (by approx 10%).

In [17]:
prompt = "For each scene in this video, generate captions that describe the scene along with any spoken text placed in quotation marks. Place each caption into an object with the timecode of the caption in the video."
media_resolution = 'MEDIA_RESOLUTION_LOW'

interaction = client.models.generate_content(
    model=MODEL_ID,
    config=types.GenerateContentConfig(
        media_resolution=media_resolution,),
    contents=[
        types.Part(file_data=types.FileData(
            file_uri="https://youtu.be/DSxM6-u2LSs?si=u8fk_rq6xj2tQDpt",
            mime_type="video/mp4" # Assuming video/mp4 is acceptable for YouTube URLs
        )),
        types.Part(text=prompt),
    ],
)

display(Markdown(interaction.text))

[
  {
    "00:00": "Close up of trees on a bright winter day. "
  },
  {
    "00:01": "The camera captures the entrance of a grave monument. "
  },
  {
    "00:02": "The shot captures the top of a stone grave. "
  },
  {
    "00:03": "The woman stands in front of the monument and sings \"why you so mad?\"."
  },
  {
    "00:05": "The woman standing in front of the monument sings \"snakes everywhere bro\"."
  },
  {
    "00:07": "The close-up of the woman's face while singing \"backfr\"."
  },
  {
    "00:08": "The woman standing in front of the monument sings \"backfromparadise\"."
  },
  {
    "00:10": "A medium shot of a woman in a red jersey and white headscarf."
  },
  {
    "00:12": "The woman standing in front of the monument sings \"I was flo\"."
  },
  {
    "00:13": "The woman dancing on a road between the monuments."
  },
  {
    "00:14": "Close up of the woman's face and teeth."
  },
  {
    "00:15": "The woman standing in front of the monument and singing \"I was floating high\"."
  },
  {
    "00:15": "The woman standing in front of the monument and singing \"hate\"."
  },
  {
    "00:16": "The woman standing in front of the monument and singing \"on another fie\"."
  },
  {
    "00:18": "The woman standing in front of the monument and singing \"think I need a\"."
  },
  {
    "00:19": "The woman dancing on a road between the monuments."
  },
  {
    "00:20": "The woman standing in front of the monument and singing \"rotten in they eyes\"."
  },
  {
    "00:21": "The woman standing in front of the monument and singing \"wishing they\"."
  },
  {
    "00:22": "A close-up of the woman singing \"rotten in they eyes\"."
  },
  {
    "00:23": "The woman dancing on a road between the monuments sings \"wishing they could\"."
  },
  {
    "00:24": "The woman standing in front of the monument and singing \"sword\"."
  },
  {
    "00:26": "The woman standing in front of the monument and singing \"serpents on attack\"."
  },
  {
    "00:27": "The woman dancing in front of the monument. "
  },
  {
    "00:28": "The woman standing in front of the monument and singing \"venom when they smile b\"."
  },
  {
    "00:30": "The woman standing in front of the monument and singing \"my halo keep me strapped\"."
  },
  {
    "00:31": "The woman dancing on a road between the monuments. "
  },
  {
    "00:32": "The woman dancing on a road between the monuments sings \"but my halo keep me strapped\"."
  },
  {
    "00:33": "The woman standing in front of the monument and singing \"but my halo keep me strapped\"."
  },
  {
    "00:34": "The woman dancing in front of the monument sings \"but my halo keep me strapped\"."
  },
  {
    "00:35": "A close up of the woman dancing. "
  },
  {
    "00:37": "The woman standing in front of the monument and singing \"I don't know why they hate on me\"."
  },
  {
    "00:39": "The woman dancing in front of the monument. "
  },
  {
    "00:40": "The close-up of the woman sings \"they can smell The Legacy\"."
  },
  {
    "00:42": "The woman dancing on a road between the monuments sings \"Uh huh\"."
  },
  {
    "00:43": "The woman standing in front of the monument and singing \"I'm standing on Supremacy\"."
  },
  {
    "00:45": "The woman standing in front of the monument and singing \"Got bitche\"."
  },
  {
    "00:46": "The woman standing in front of the monument and singing \"Got bitches tryna come for me\"."
  },
  {
    "00:47": "The woman standing in front of the monument and singing \"na come for me\"."
  },
  {
    "00:48": "The woman standing in front of the monument and singing \"Oh you wan love me now?\"."
  },
  {
    "00:49": "The woman standing in front of the monument and singing \"Diamonds gon make them vow\"."
  },
  {
    "00:50": "The woman standing in front of the monument and singing \"Emeralds\"."
  },
  {
    "00:51": "The woman standing in front of the monument and singing \"Emeralds in my cortex\"."
  },
  {
    "00:52": "The woman standing in front of the monument and singing \"I don't hold no grudge no\"."
  },
  {
    "00:53": "The woman standing in front of the monument and singing \"just dance on me\"."
  },
  {
    "00:54": "The woman standing in front of the monument and singing \"right right righ\"."
  },
  {
    "00:55": "The woman standing in front of the monument and singing \"got guns on me\"."
  },
  {
    "00:56": "The woman standing in front of the monument and singing \"right right\"."
  },
  {
    "00:57": "A medium shot of an angel statue. "
  },
  {
    "00:58": "A close up of the woman sings \"right\"."
  },
  {
    "00:59": "The close-up of the woman sings \"I don't know why they hate on me\"."
  },
  {
    "01:01": "The woman dancing in front of the monument. "
  },
  {
    "01:03": "The close-up of the woman sings \"they can smell the Legacy\"."
  },
  {
    "01:04": "The woman dancing on a road between the monuments. "
  },
  {
    "01:05": "The woman standing in front of the monument and singing \"I'm standing on Supremacy\"."
  },
  {
    "01:07": "The woman dancing in front of the monument. "
  },
  {
    "01:08": "The woman standing in front of the monument and singing \"Got bitches tryna come for me\"."
  },
  {
    "01:09": "The woman dancing in front of the monument. "
  },
  {
    "01:10": "A close up of the woman singing \"Got bitches tryna come for me\"."
  },
  {
    "01:11": "The woman standing in front of the monument and singing \"I was floating high\"."
  },
  {
    "01:12": "The woman standing in front of the monument and singing \"jealous bitches hate on me\"."
  },
  {
    "01:13": "The woman standing in front of the monument and singing \"on another field\"."
  },
  {
    "01:14": "The woman dancing on a road between the monuments sings \"think I need a pedigree\"."
  },
  {
    "01:16": "The woman standing in front of the monument and singing \"rotten\"."
  },
  {
    "01:17": "The woman standing in front of the monument and singing \"rotten in they eyes\"."
  },
  {
    "01:18": "The woman standing in front of the monument and singing \"wishing they could silence me\"."
  },
  {
    "01:19": "The woman dancing on a road between the monuments. "
  },
  {
    "01:20": "The woman dancing on a road between the monuments sings \"they could never silence me\"."
  },
  {
    "01:21": "The woman standing in front of the monument and singing \"swords on my back\"."
  },
  {
    "01:22": "The woman standing in front of the monument and singing \"serpents on\"."
  },
  {
    "01:23": "The woman standing in front of the monument and singing \"serpents on attack\"."
  },
  {
    "01:24": "The woman standing in front of the monument and singing \"venom when they smile\"."
  },
  {
    "01:25": "The woman standing in front of the monument and singing \"but my halo keep me strapped\"."
  },
  {
    "01:27": "The woman dancing on a road between the monuments. "
  },
  {
    "01:28": "The woman dancing on a road between the monuments sings \"but my halo keep me strapped\"."
  },
  {
    "01:30": "The woman dancing in front of the monument. "
  },
  {
    "01:31": "The close-up of the woman singing \"but my halo keep me strapp\"."
  },
  {
    "01:32": "The woman dancing in front of the monument. "
  },
  {
    "01:33": "A close up of the woman dancing sings \"I don\"."
  },
  {
    "01:34": "The woman standing in front of the monument and singing \"I don't know why they hate on me\"."
  },
  {
    "01:35": "The close-up of the woman singing \"I don't know why they hate on me\"."
  },
  {
    "01:36": "The woman dancing on a road between the monuments sings \"they can smell the Legacy\"."
  },
  {
    "01:37": "A close up of the woman singing \"they can smell the Legacy\"."
  },
  {
    "01:38": "The woman dancing on a road between the monuments. "
  },
  {
    "01:39": "The woman standing in front of the monument and singing \"I'm standing on Supremacy\"."
  },
  {
    "01:40": "The woman standing in front of the monument and singing \"I'm standing on Supremacy\"."
  },
  {
    "01:41": "The woman dancing in front of the monument. "
  },
  {
    "01:42": "The woman standing in front of the monument and singing \"got bitches tryna come for me\"."
  },
  {
    "01:43": "The woman dancing in front of the monument. "
  },
  {
    "01:44": "The close-up of the woman singing \"got bitches tryna come for me\"."
  },
  {
    "01:45": "A medium shot of a woman drinking water."
  },
  {
    "01:46": "The woman dancing on a road between the monuments. "
  },
  {
    "01:47": "A close up of the woman dancing sings \"sh\"."
  },
  {
    "01:48": "The woman dancing on a road between the monuments sings \"sh show me you would ride for me\"."
  },
  {
    "01:49": "The woman dancing in front of the monument. "
  },
  {
    "01:50": "The woman standing in front of the monument and singing \"got bitches\"."
  },
  {
    "01:51": "The woman standing in front of the monument and singing \"got bitches tryna come for me\"."
  },
  {
    "01:52": "The woman dancing on a road between the monuments sings \"got bitches tryna come for me\"."
  },
  {
    "01:53": "The woman standing in front of the monument and singing \"tell me\"."
  },
  {
    "01:54": "The woman standing in front of the monument and singing \"tell me you would ride for me\"."
  },
  {
    "01:55": "The woman dancing in front of the monument. "
  },
  {
    "01:56": "The close-up of the woman singing \"bad\"."
  },
  {
    "01:57": "The woman dancing in front of the monument. "
  },
  {
    "01:58": "The woman dancing on a road between the monuments. "
  },
  {
    "02:00": "A close up of the woman dancing. "
  },
  {
    "02:01": "The close-up of the woman singing \"bad\"."
  },
  {
    "02:02": "The woman dancing on a road between the monuments. "
  },
  {
    "02:04": "The camera captures the top of a stone grave. "
  },
  {
    "02:05": "The shot captures the top of a monument sings \"why you so mad\"."
  },
  {
    "02:06": "The woman dancing on a road between the monuments. "
  },
  {
    "02:07": "A medium shot of an angel statue holding a book sings \"snakes everywhere bro\"."
  },
  {
    "02:08": "A medium shot of an angel statue holding a flower wreath sings \"snakes everywhere bro\"."
  },
  {
    "02:09": "A medium shot of an angel statue holding a book. "
  },
  {
    "02:10": "The shot captures the top of a stone grave. "
  },
  {
    "02:11": "The camera captures the entrance of a grave monument. "
  }
]